# Build Document Profile Graph

Notebook nay chi dung de dung lai 2 file graph tu:
- `document_train_users.csv`
- `document_train_documents.csv`

Output:
- `document_graph_nodes.csv`
- `document_graph_edges.csv`

Phien ban nay chi dung cac thuoc tinh:
- `major`
- `school`
- `cohort`

In [1]:
from __future__ import annotations

from pathlib import Path
import re
import unicodedata

import pandas as pd

In [2]:
DATA_DIR = Path.cwd()

USERS_FILE = DATA_DIR / 'document_train_users.csv'
DOCUMENTS_FILE = DATA_DIR / 'document_train_documents.csv'

NODES_OUTPUT = DATA_DIR / 'document_graph_nodes.csv'
EDGES_OUTPUT = DATA_DIR / 'document_graph_edges.csv'

In [3]:
users_df = pd.read_csv(USERS_FILE)
documents_df = pd.read_csv(DOCUMENTS_FILE)

print('users    :', len(users_df))
print('documents:', len(documents_df))

users_df.head()

users    : 4121
documents: 152


,userId,email,displayName,interests,school,major,cohort,role,status,profileVisibility
0,00011052-b71e-5d93-a12d-b4e8b200fbdf,fb_3221@facebook-combined.local,Trần Thị Hương,"Quan tâm đến bạn bè, cộng đồng và những hoạt đ...",Đại học Quy Nhơn,Giáo dục tiểu học,K33,USER,ACTIVE,PUBLIC
1,00070f68-8cae-590f-aad0-ce7bf65d54e5,fb_901@facebook-combined.local,Trần Đức Châu,"Quan tâm đến bạn bè, cộng đồng và những hoạt đ...",Đại học Khoa học Xã hội và Nhân văn - ĐHQGHN,Quan hệ công chúng,K38,USER,ACTIVE,PUBLIC
2,001409a7-980b-5ecf-9f54-2a8c2953bf6c,fb_2331@facebook-combined.local,Hồ Gia Hải,"Ưu tiên các mối quan hệ chân thành, giao tiếp ...",Đại học Nông Lâm TP.HCM,Chăn nuôi,K35,USER,ACTIVE,PUBLIC
3,0028d64f-183d-53f5-8017-7713f18eb0da,fb_882@facebook-combined.local,Lê Bảo Châu,"Thích trò chuyện, làm việc nhóm và lan tỏa nhữ...",Đại học Khoa học Xã hội và Nhân văn - ĐHQGHN,Quan hệ công chúng,K41,USER,ACTIVE,PUBLIC
4,0031cb49-1320-525a-8e72-3abf339f05ea,fb_430@facebook-combined.local,Đỗ Quang Bình,"Yêu thích môi trường cởi mở, nơi mọi người có ...",Học viện Báo chí và Tuyên truyền,Truyền thông đa phương tiện,K34,USER,ACTIVE,PUBLIC


In [4]:
documents_df.head()

,documentId,title,fileName,fileType,subject,school,major,cohort,description,tags,visibility,status,viewsCount,downloadsCount,uploaderId,createdAt,updatedAt,sourceType
0,7bb9d9b8-c63a-4c92-a097-23a236e5120f,K-Means From Theory to Practice (3),K-Means From Theory to Practice (3).pdf,PDF,ML,ĐH Quy Nhơn,CNTT,K45,NaN,NaN,PUBLIC,ACTIVE,1,0,2901fc71-a8a0-4634-a6af-6a0d980430d7,2026-04-27T07:56:33.611Z,2026-05-11T07:02:58.935Z,NaN
1,56ebfdfe-13dc-473c-bdea-ab78970246f8,Báo cáo thực tập-LeXuanNgoc,BÃ¡o cÃ¡o thá»±c táº­p-LeXuanNgoc.doc,DOC,ML,ĐH Quy Nhơn,CNTT,K45,NaN,NaN,PUBLIC,ACTIVE,1,0,2901fc71-a8a0-4634-a6af-6a0d980430d7,2026-05-11T07:02:45.309Z,2026-05-11T07:16:23.513Z,NaN
2,1756f62b-88c5-41f8-aa54-a21bac2a86a3,Thủy sản căn bản,Thủy sản căn bản.docx,DOC,Nông nghiệp,Học viện Báo chí và Tuyên truyền,Chăn nuôi,K30,"Tài liệu học tập về thủy sản căn bản, hỗ trợ s...",thủy sản|nông nghiệp|chăn nuôi|hữu cơ,PUBLIC,ACTIVE,0,0,8792b584-b835-4340-889b-06f1ae962e99,2026-05-16T08:31:19.020Z,2026-05-16T08:31:19.020Z,NaN
3,35976ac5-1a40-4616-a15b-b6dbd7737262,Biên dịch văn bản chuyên ngành,Biên dịch văn bản chuyên ngành.pptx,PPT,Ngoại ngữ,Học viện Công nghệ Bưu chính Viễn thông,Biên phiên dịch,K33,Tài liệu học tập về biên dịch văn bản chuyên n...,học thuật|ngoại ngữ|biên dịch|giao tiếp,PUBLIC,ACTIVE,0,0,7874ff7c-813c-5627-b476-244d84950793,2026-05-16T08:31:20.901Z,2026-05-16T08:31:20.901Z,NaN
4,ecb0c11d-7693-4815-b441-e1ebe2b96b3d,Luật lao động,Luật lao động.pptx,PPT,Luật,Học viện Ngân hàng,Luật dân sự,K41,"Tài liệu học tập về luật lao động, hỗ trợ sinh...",lao động|dân sự|hợp đồng|luật,PUBLIC,ACTIVE,0,0,64353153-04ad-5366-8bb8-22e1d3ce53d4,2026-05-16T08:31:21.857Z,2026-05-16T08:31:21.857Z,NaN


In [5]:
def normalize_text(value: object) -> str:
    if pd.isna(value):
        return ''

    text = str(value).strip().lower()
    text = unicodedata.normalize('NFKC', text)
    text = ''.join(
        char for char in unicodedata.normalize('NFD', text)
        if unicodedata.category(char) != 'Mn'
    )
    text = re.sub(r'[^a-z0-9]+', '_', text)
    return text.strip('_')


def build_attribute_node_id(prefix: str, value: object) -> str | None:
    normalized = normalize_text(value)
    if not normalized:
        return None
    return f'{prefix}_{normalized}'

In [6]:
profile_users_df = users_df[['userId', 'displayName', 'school', 'major', 'cohort']].copy()
profile_documents_df = documents_df[['documentId', 'title', 'school', 'major', 'cohort']].copy()

for field in ['school', 'major', 'cohort']:
    profile_users_df[f'norm_{field}'] = profile_users_df[field].map(normalize_text)
    profile_documents_df[f'norm_{field}'] = profile_documents_df[field].map(normalize_text)

profile_users_df.head()

,userId,displayName,school,major,cohort,norm_school,norm_major,norm_cohort
0,00011052-b71e-5d93-a12d-b4e8b200fbdf,Trần Thị Hương,Đại học Quy Nhơn,Giáo dục tiểu học,K33,ai_hoc_quy_nhon,giao_duc_tieu_hoc,k33
1,00070f68-8cae-590f-aad0-ce7bf65d54e5,Trần Đức Châu,Đại học Khoa học Xã hội và Nhân văn - ĐHQGHN,Quan hệ công chúng,K38,ai_hoc_khoa_hoc_xa_hoi_va_nhan_van_hqghn,quan_he_cong_chung,k38
2,001409a7-980b-5ecf-9f54-2a8c2953bf6c,Hồ Gia Hải,Đại học Nông Lâm TP.HCM,Chăn nuôi,K35,ai_hoc_nong_lam_tp_hcm,chan_nuoi,k35
3,0028d64f-183d-53f5-8017-7713f18eb0da,Lê Bảo Châu,Đại học Khoa học Xã hội và Nhân văn - ĐHQGHN,Quan hệ công chúng,K41,ai_hoc_khoa_hoc_xa_hoi_va_nhan_van_hqghn,quan_he_cong_chung,k41
4,0031cb49-1320-525a-8e72-3abf339f05ea,Đỗ Quang Bình,Học viện Báo chí và Tuyên truyền,Truyền thông đa phương tiện,K34,hoc_vien_bao_chi_va_tuyen_truyen,truyen_thong_a_phuong_tien,k34


In [7]:
profile_documents_df.head()

,documentId,title,school,major,cohort,norm_school,norm_major,norm_cohort
0,7bb9d9b8-c63a-4c92-a097-23a236e5120f,K-Means From Theory to Practice (3),ĐH Quy Nhơn,CNTT,K45,h_quy_nhon,cntt,k45
1,56ebfdfe-13dc-473c-bdea-ab78970246f8,Báo cáo thực tập-LeXuanNgoc,ĐH Quy Nhơn,CNTT,K45,h_quy_nhon,cntt,k45
2,1756f62b-88c5-41f8-aa54-a21bac2a86a3,Thủy sản căn bản,Học viện Báo chí và Tuyên truyền,Chăn nuôi,K30,hoc_vien_bao_chi_va_tuyen_truyen,chan_nuoi,k30
3,35976ac5-1a40-4616-a15b-b6dbd7737262,Biên dịch văn bản chuyên ngành,Học viện Công nghệ Bưu chính Viễn thông,Biên phiên dịch,K33,hoc_vien_cong_nghe_buu_chinh_vien_thong,bien_phien_dich,k33
4,ecb0c11d-7693-4815-b441-e1ebe2b96b3d,Luật lao động,Học viện Ngân hàng,Luật dân sự,K41,hoc_vien_ngan_hang,luat_dan_su,k41


In [8]:
nodes = []
edges = []
seen_nodes = set()


def add_node(node_id: str, node_type: str, entity_id: str, label: str) -> None:
    if node_id in seen_nodes:
        return
    seen_nodes.add(node_id)
    nodes.append({
        'nodeId': node_id,
        'nodeType': node_type,
        'entityId': entity_id,
        'label': label,
    })


def add_edge(source: str, target: str, edge_type: str, weight: float) -> None:
    edges.append({
        'source': source,
        'target': target,
        'edgeType': edge_type,
        'weight': weight,
    })

In [9]:
for row in profile_users_df.itertuples(index=False):
    user_node_id = f'U_{row.userId}'
    add_node(
        node_id=user_node_id,
        node_type='USER',
        entity_id=str(row.userId),
        label=row.displayName if pd.notna(row.displayName) else str(row.userId),
    )

    attribute_specs = [
        ('M', 'MAJOR', row.major, 2.0, 'HAS_MAJOR'),
        ('S', 'SCHOOL', row.school, 1.0, 'STUDIES_AT'),
        ('C', 'COHORT', row.cohort, 1.0, 'BELONGS_TO_COHORT'),
    ]

    for prefix, node_type, raw_value, weight, edge_type in attribute_specs:
        attribute_node_id = build_attribute_node_id(prefix, raw_value)
        if not attribute_node_id:
            continue

        add_node(
            node_id=attribute_node_id,
            node_type=node_type,
            entity_id=normalize_text(raw_value),
            label=str(raw_value),
        )
        add_edge(user_node_id, attribute_node_id, edge_type, weight)

In [10]:
for row in profile_documents_df.itertuples(index=False):
    document_node_id = f'D_{row.documentId}'
    add_node(
        node_id=document_node_id,
        node_type='DOCUMENT',
        entity_id=str(row.documentId),
        label=row.title if pd.notna(row.title) else str(row.documentId),
    )

    attribute_specs = [
        ('M', 'MAJOR', row.major, 2.0, 'FOR_MAJOR'),
        ('S', 'SCHOOL', row.school, 1.0, 'FROM_SCHOOL'),
        ('C', 'COHORT', row.cohort, 1.0, 'FOR_COHORT'),
    ]

    for prefix, node_type, raw_value, weight, edge_type in attribute_specs:
        attribute_node_id = build_attribute_node_id(prefix, raw_value)
        if not attribute_node_id:
            continue

        add_node(
            node_id=attribute_node_id,
            node_type=node_type,
            entity_id=normalize_text(raw_value),
            label=str(raw_value),
        )
        add_edge(document_node_id, attribute_node_id, edge_type, weight)

In [11]:
nodes_df = pd.DataFrame(nodes).sort_values(['nodeType', 'nodeId']).reset_index(drop=True)
edges_df = pd.DataFrame(edges).sort_values(['edgeType', 'source', 'target']).reset_index(drop=True)

print('graph nodes:', len(nodes_df))
print('graph edges:', len(edges_df))

display(nodes_df['nodeType'].value_counts().rename_axis('nodeType').reset_index(name='count'))
display(edges_df['edgeType'].value_counts().rename_axis('edgeType').reset_index(name='count'))

graph nodes: 4375
graph edges: 12815


,nodeType,count
0,USER,4121
1,DOCUMENT,152
2,MAJOR,45
3,SCHOOL,41
4,COHORT,16


,edgeType,count
0,BELONGS_TO_COHORT,4121
1,HAS_MAJOR,4121
2,STUDIES_AT,4121
3,FOR_COHORT,152
4,FROM_SCHOOL,152
5,FOR_MAJOR,148


In [12]:
nodes_df.to_csv(NODES_OUTPUT, index=False, encoding='utf-8')
edges_df.to_csv(EDGES_OUTPUT, index=False, encoding='utf-8')

print('saved nodes ->', NODES_OUTPUT.name)
print('saved edges ->', EDGES_OUTPUT.name)

saved nodes -> document_graph_nodes.csv
saved edges -> document_graph_edges.csv


In [13]:
nodes_df.head(20)

,nodeId,nodeType,entityId,label
0,C_k30,COHORT,k30,K30
1,C_k31,COHORT,k31,K31
2,C_k32,COHORT,k32,K32
3,C_k33,COHORT,k33,K33
4,C_k34,COHORT,k34,K34
5,C_k35,COHORT,k35,K35
6,C_k36,COHORT,k36,K36
7,C_k37,COHORT,k37,K37
8,C_k38,COHORT,k38,K38
9,C_k39,COHORT,k39,K39


In [14]:
edges_df.head(1000)

,source,target,edgeType,weight
0,U_00011052-b71e-5d93-a12d-b4e8b200fbdf,C_k33,BELONGS_TO_COHORT,1.0
1,U_00070f68-8cae-590f-aad0-ce7bf65d54e5,C_k38,BELONGS_TO_COHORT,1.0
2,U_001409a7-980b-5ecf-9f54-2a8c2953bf6c,C_k35,BELONGS_TO_COHORT,1.0
3,U_0028d64f-183d-53f5-8017-7713f18eb0da,C_k41,BELONGS_TO_COHORT,1.0
4,U_0031cb49-1320-525a-8e72-3abf339f05ea,C_k34,BELONGS_TO_COHORT,1.0
...,...,...,...,...
995,U_3e6d3872-ad4a-57bb-a4f5-ab25917886da,C_k35,BELONGS_TO_COHORT,1.0
996,U_3e99e002-52e5-5e8b-93a1-03204e78d437,C_k38,BELONGS_TO_COHORT,1.0
997,U_3e9b47eb-1d65-5584-8196-02f18d64f648,C_k41,BELONGS_TO_COHORT,1.0
998,U_3e9d1f82-5e2f-52fe-ab1c-7347d309c6e2,C_k35,BELONGS_TO_COHORT,1.0
